In [ ]:
from google.colab import drive

drive.mount("/content/drive")

from pathlib import Path
from datetime import datetime

import os
import sys
import json
import shutil
import hashlib
import subprocess
import pandas as pd

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/AI/Project/TrafficGuard-AI"
)

assert PROJECT_ROOT.is_dir(), "Project directory not found."

SESSION_ID = datetime.now().strftime("%Y%m%d-%H%M%S")

PHASE11_DIR = (
    PROJECT_ROOT / "results/final_system_testing" / SESSION_ID
)

PHASE11_DIR.mkdir(parents=True, exist_ok=True)

def sha256_file(path):

    sha = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(lambda: f.read(1024**2), b""):
            sha.update(chunk)

    return sha.hexdigest()

print("Project:", PROJECT_ROOT)
print("Phase 11 results:", PHASE11_DIR)

print("\nPHASE 11 ENVIRONMENT READY")

In [ ]:
REQUIREMENTS = PROJECT_ROOT / "requirements_phase10.txt"

assert REQUIREMENTS.is_file()

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        str(REQUIREMENTS)
    ],
    check=True
)

import torch
import cv2
import numpy as np
import streamlit
import ultralytics

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Ultralytics:", ultralytics.__version__)
print("Streamlit:", streamlit.__version__)
print("OpenCV:", cv2.__version__)
print("GPU available:", torch.cuda.is_available())

print("\nDEPENDENCIES VERIFIED")

In [ ]:
REQUIRED_FILES = [

    "app/streamlit_app.py",

    "src/trafficguard_engine.py",
    "src/trafficguard_refined.py",
    "src/video_processor.py",

    "configs/app_config.json",

    "models/helmet/helmet_best.pt",
    "models/seatbelt/seatbelt_best.pt",
    "models/context/vehicle_context_yolov8n.pt",

    ".streamlit/config.toml",

    "requirements_phase10.txt",

    "results/final_evaluation/tables/independent_test_metrics.csv",

    "results/final_evaluation/reports/final_inference_config.json"
]

audit_records = []

for relative in REQUIRED_FILES:

    path = PROJECT_ROOT / relative

    exists = path.is_file()

    audit_records.append({

        "File": relative,

        "Exists": exists,

        "Size MB": round(
            path.stat().st_size / 1024**2, 4
        ) if exists else 0,

        "SHA256": sha256_file(path) if exists else None

    })

audit_df = pd.DataFrame(audit_records)

display(audit_df.drop(columns=["SHA256"]))

audit_df.to_csv(
    PHASE11_DIR / "original_project_audit.csv",
    index=False
)

assert audit_df["Exists"].all(), "Required files are missing."

ORIGINAL_HASHES = dict(
    zip(audit_df["File"], audit_df["SHA256"])
)

print("\nALL REQUIRED PROJECT FILES FOUND")

In [ ]:
APP_CONFIG = json.loads(
    (PROJECT_ROOT / "configs/app_config.json").read_text()
)

model_records = []

for task, cfg in APP_CONFIG["models"].items():

    path = PROJECT_ROOT / cfg["weights"]

    actual_hash = sha256_file(path)

    verified = actual_hash == cfg["model_sha256"]

    model_records.append({

        "Model": task,

        "Architecture": cfg.get(
            "architecture", "YOLOv8n"
        ),

        "Confidence": cfg["confidence"],

        "Size MB": round(
            path.stat().st_size / 1024**2, 2
        ),

        "Integrity Verified": verified

    })

models_df = pd.DataFrame(model_records)

display(models_df)

assert models_df["Integrity Verified"].all()

models_df.to_csv(
    PHASE11_DIR / "final_model_integrity.csv",
    index=False
)

print("\nALL THREE MODEL CHECKPOINTS VERIFIED")

In [ ]:
import ast

source_files = [

    "src/trafficguard_engine.py",

    "src/trafficguard_refined.py",

    "src/video_processor.py",

    "app/streamlit_app.py"
]

syntax_records = []

for relative in source_files:

    path = PROJECT_ROOT / relative

    try:

        ast.parse(path.read_text(encoding="utf-8"))

        status = "PASS"
        error = ""

    except Exception as e:

        status = "FAIL"
        error = str(e)

    syntax_records.append({

        "File": relative,
        "Status": status,
        "Error": error

    })

syntax_df = pd.DataFrame(syntax_records)

display(syntax_df)

assert (syntax_df["Status"] == "PASS").all()

syntax_df.to_csv(
    PHASE11_DIR / "syntax_testing.csv",
    index=False
)

print("\nSOURCE CODE SYNTAX TESTING PASSED")

In [ ]:
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from trafficguard_refined import (
    RefinedTrafficGuard,
    PROFILES
)

from video_processor import process_video

DEVICE = 0 if torch.cuda.is_available() else "cpu"

engine = RefinedTrafficGuard(

    project_root=PROJECT_ROOT,

    device=DEVICE,

    profile="standard"

)

print("Device:", DEVICE)
print("Available profiles:", list(PROFILES.keys()))

for task, model in engine.models.items():

    print("\nModel:", task)

    if task != "context":
        print("Classes:", model.names)

print("\nFINAL INFERENCE ENGINE LOADED")

In [ ]:
import time

TEST_RESULTS = []

def run_test(name, function):

    start = time.perf_counter()

    try:

        function()

        status = "PASS"
        error = ""

    except Exception as e:

        status = "FAIL"
        error = str(e)

    TEST_RESULTS.append({

        "Test": name,

        "Status": status,

        "Execution ms": round(
            (time.perf_counter() - start) * 1000, 2
        ),

        "Error": error

    })


BLACK = np.zeros(
    (640, 640, 3),
    dtype=np.uint8
)

WHITE = np.full(
    (640, 640, 3),
    255,
    dtype=np.uint8
)


def test_model_loading():

    assert set(engine.models.keys()) == {
        "helmet", "seatbelt", "context"
    }


def test_image_mode(mode):

    result = engine.process_image(
        BLACK,
        mode=mode
    )

    assert "summary" in result
    assert "detections" in result
    assert "annotated_image" in result

    assert result["annotated_image"].shape == BLACK.shape

    assert result["summary"]["mode"] == mode


def test_blank_image(image):

    result = engine.process_image(image)

    assert result["summary"]["total_detections"] == 0


def test_invalid_image():

    try:

        engine.process_image(None)

    except ValueError:

        return

    raise AssertionError("Invalid image was accepted")


def test_invalid_mode():

    try:

        engine.process_image(
            BLACK,
            mode="invalid"
        )

    except ValueError:

        return

    raise AssertionError("Invalid mode was accepted")


def test_invalid_profile():

    try:

        engine.set_profile("invalid_profile")

    except ValueError:

        return

    raise AssertionError("Invalid profile was accepted")


def test_profile(profile):

    engine.set_profile(profile)

    result = engine.process_image(
        BLACK,
        mode="helmet"
    )

    assert result["summary"]["profile"] == profile

    engine.set_profile("standard")


def test_json_export():

    result = engine.process_image(BLACK)

    json.dumps(result["summary"])

    json.dumps(result["detections"])


def test_csv_export():

    result = engine.process_image(BLACK)

    df = pd.DataFrame([result["summary"]])

    path = PHASE11_DIR / "csv_export_test.csv"

    df.to_csv(path, index=False)

    loaded = pd.read_csv(path)

    assert len(loaded) == 1

    assert "total_detections" in loaded.columns


def test_configuration():

    assert engine.config["models"]["helmet"]["confidence"] == 0.30

    assert engine.config["models"]["seatbelt"]["confidence"] == 0.35


run_test("Load all three models", test_model_loading)

run_test(
    "Helmet detection mode",
    lambda: test_image_mode("helmet")
)

run_test(
    "Seatbelt detection mode",
    lambda: test_image_mode("seatbelt")
)

run_test(
    "Combined detection mode",
    lambda: test_image_mode("combined")
)

run_test(
    "Black image negative test",
    lambda: test_blank_image(BLACK)
)

run_test(
    "White image negative test",
    lambda: test_blank_image(WHITE)
)

run_test("Invalid image handling", test_invalid_image)

run_test("Invalid mode handling", test_invalid_mode)

run_test("Invalid profile handling", test_invalid_profile)

for profile in PROFILES:

    run_test(
        f"Profile: {profile}",
        lambda p=profile: test_profile(p)
    )

engine.set_profile("standard")

run_test("JSON export", test_json_export)

run_test("CSV export", test_csv_export)

run_test("Frozen threshold verification", test_configuration)

test_df = pd.DataFrame(TEST_RESULTS)

display(test_df)

test_df.to_csv(
    PHASE11_DIR / "automated_system_tests.csv",
    index=False
)

print("\nPassed:", (test_df["Status"] == "PASS").sum())

print("Failed:", (test_df["Status"] == "FAIL").sum())

print("Total:", len(test_df))

In [ ]:
VIDEO_TEST_DIR = PHASE11_DIR / "video_test"

VIDEO_TEST_DIR.mkdir(exist_ok=True)

TEST_VIDEO = VIDEO_TEST_DIR / "synthetic_input.mp4"

TEST_OUTPUT = VIDEO_TEST_DIR / "synthetic_output.mp4"

writer = cv2.VideoWriter(

    str(TEST_VIDEO),

    cv2.VideoWriter_fourcc(*"mp4v"),

    10,

    (320, 320)

)

assert writer.isOpened()

for index in range(20):

    frame = np.zeros(
        (320, 320, 3),
        dtype=np.uint8
    )

    cv2.putText(

        frame,

        f"Frame {index}",

        (20, 40),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.6,

        (255, 255, 255),

        1

    )

    writer.write(frame)

writer.release()

engine.set_profile("standard")

TEST_OUTPUT = VIDEO_TEST_DIR / (
    f"output_{datetime.now():%H%M%S%f}.mp4"
)

video_test_report = process_video(

    engine=engine,

    input_path=TEST_VIDEO,

    output_path=TEST_OUTPUT,

    mode="combined",

    frame_stride=5,

    max_frames=None

)

assert TEST_OUTPUT.is_file()

assert TEST_OUTPUT.with_suffix(".csv").is_file()

assert TEST_OUTPUT.with_suffix(".json").is_file()

assert video_test_report["processed_frames"] == 4

print(json.dumps(video_test_report, indent=4))

print("\nVIDEO PROCESSING FUNCTIONAL TEST PASSED")

In [ ]:
from pathlib import Path
import time

# Use existing PROJECT_ROOT and SESSION_ID if already defined in an earlier cell
if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = Path("/content/drive/MyDrive/AI/Project/TrafficGuard-AI")

if "SESSION_ID" not in globals():
    SESSION_ID = time.strftime("%Y%m%d_%H%M%S")

RELEASE_ROOT = Path(f"/content/TrafficGuard-AI-release-{SESSION_ID}")

assert not RELEASE_ROOT.exists(), f"Release folder already exists: {RELEASE_ROOT}"
RELEASE_ROOT.mkdir(parents=True)

folders = [
    "app",
    "src",
    "configs",
    "models/helmet",
    "models/seatbelt",
    "models/context",
    "notebooks",
    "datasets",
    "docs",
    "tests",
    "results/final_evaluation/tables",
    "results/final_evaluation/figures",
    "results/final_evaluation/reports",
    "results/system_testing",
    ".streamlit",
]

for folder in folders:
    (RELEASE_ROOT / folder).mkdir(parents=True, exist_ok=True)

print("Master project:", PROJECT_ROOT)
print("GitHub release:", RELEASE_ROOT)
print("\nSEPARATE RELEASE DIRECTORY CREATED")

In [ ]:
for folder in folders:
    (RELEASE_ROOT / folder).mkdir(parents=True, exist_ok=True)

In [ ]:
!ls -la "{RELEASE_ROOT}"
!find "{RELEASE_ROOT}" -maxdepth 3

In [ ]:
RELEASE_FILES = [

    "app/streamlit_app.py",

    "src/trafficguard_engine.py",

    "src/trafficguard_refined.py",

    "src/video_processor.py",

    "configs/app_config.json",

    "models/helmet/helmet_best.pt",

    "models/seatbelt/seatbelt_best.pt",

    "models/context/vehicle_context_yolov8n.pt",

    ".streamlit/config.toml",

    "requirements_phase10.txt"
]

copy_records = []

for relative in RELEASE_FILES:

    source = PROJECT_ROOT / relative

    destination = RELEASE_ROOT / relative

    assert source.is_file(), source

    destination.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    shutil.copy2(source, destination)

    verified = (
        sha256_file(source)
        == sha256_file(destination)
    )

    assert verified

    copy_records.append({

        "File": relative,

        "Size MB": round(
            destination.stat().st_size / 1024**2, 3
        ),

        "Verified": verified

    })

copy_df = pd.DataFrame(copy_records)

display(copy_df)

print("\nRELEASE ASSETS COPIED AND VERIFIED")

In [ ]:
EVALUATION_FILES = [

    "tables/independent_test_metrics.csv",

    "tables/independent_test_class_metrics.csv",

    "tables/deployment_test_metrics.csv",

    "tables/deployment_class_metrics.csv",

    "tables/selected_validation_thresholds.csv",

    "tables/validation_vs_independent_test.csv",

    "tables/inference_speed_benchmark.csv",

    "figures/confidence_optimization.png",

    "figures/validation_vs_independent_test.png",

    "figures/helmet_final_confusion_matrices.png",

    "figures/seatbelt_final_confusion_matrices.png",

    "reports/final_inference_config.json",

    "reports/final_evaluation_summary.json",

    "reports/final_evaluation_report.txt"
]

source_root = (
    PROJECT_ROOT / "results/final_evaluation"
)

destination_root = (
    RELEASE_ROOT / "results/final_evaluation"
)

for relative in EVALUATION_FILES:

    source = source_root / relative

    destination = destination_root / relative

    if not source.is_file():

        print("[MISSING]", relative)

        continue

    destination.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    shutil.copy2(source, destination)

    print("[COPIED]", relative)


for filename in [

    "automated_system_tests.csv",

    "syntax_testing.csv",

    "final_model_integrity.csv"

]:

    source = PHASE11_DIR / filename

    if source.is_file():

        shutil.copy2(

            source,

            RELEASE_ROOT / "results/system_testing" / filename

        )

print("\nSELECTED RESEARCH RESULTS COPIED")